# Netflix Content Performance & Release Strategy Analysis

## Business Objective

Netflix invests billions of dollars annually in producing and acquiring content. Understanding the factors that influence audience engagement is critical for maximizing return on investment.

This project analyzes Netflix's official 2023 engagement dataset to identify patterns in viewership across content types, languages, and release timing. The analysis aims to generate actionable business insights that can support future content strategy decisions.

In [23]:
# Import libraries for data manipulation, visualization, and statistical analysis
import pandas as pd
import numpy as np

import plotly.express as px
import plotly.graph_objects as go

from scipy.stats import ttest_ind

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 10)

In [24]:
# Load dataset
df = pd.read_csv("netflix_content.csv")

# Display first five rows
df.head()

,Title,Available Globally?,Release Date,Hours Viewed,Language Indicator,Content Type
0,The Night Agent: Season 1,Yes,2023-03-23,"81,21,00,000",English,Show
1,Ginny & Georgia: Season 2,Yes,2023-01-05,"66,51,00,000",English,Show
2,The Glory: Season 1 // 더 글로리: 시즌 1,Yes,2022-12-30,"62,28,00,000",Korean,Show
3,Wednesday: Season 1,Yes,2022-11-23,"50,77,00,000",English,Show
4,Queen Charlotte: A Bridgerton Story,Yes,2023-05-04,"50,30,00,000",English,Movie


## Data Cleaning

The dataset contains inconsistent data types and duplicate records that must be addressed before analysis.

The following cleaning steps are performed:

- Convert **Hours Viewed** from text to numeric format.
- Convert **Release Date** to datetime format.
- Remove duplicate records.
- Retain missing release dates, as they likely represent older or licensed content rather than data quality issues.

In [25]:
# Clean and prepare the dataset for analysis

# Create a copy of the original dataset
df = df.copy()

# Convert Hours Viewed from string to integer
df["Hours Viewed"] = (
    df["Hours Viewed"]
    .str.replace(",", "", regex=False)
    .astype(int)
)

# Convert Release Date to datetime
df["Release Date"] = pd.to_datetime(
    df["Release Date"],
    errors="coerce"
)

# Remove duplicate rows
df = df.drop_duplicates()

# Display updated dataset information
print("=" * 50)
print("Updated Dataset Information")
print("=" * 50)
df.info()

print("\n")

print("=" * 50)
print("Remaining Missing Values")
print("=" * 50)
display(df.isnull().sum())

Updated Dataset Information
<class 'pandas.core.frame.DataFrame'>
Index: 24345 entries, 0 to 24807
Data columns (total 6 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   Title                24345 non-null  object        
 1   Available Globally?  24345 non-null  object        
 2   Release Date         7975 non-null   datetime64[ns]
 3   Hours Viewed         24345 non-null  int32         
 4   Language Indicator   24345 non-null  object        
 5   Content Type         24345 non-null  object        
dtypes: datetime64[ns](1), int32(1), object(4)
memory usage: 1.2+ MB


Remaining Missing Values


Title                      0
Available Globally?        0
Release Date           16370
Hours Viewed               0
Language Indicator         0
Content Type               0
dtype: int64

## Feature Engineering

Additional features are created from the release date to enable temporal analysis of Netflix content performance.

The engineered features include:

- Release Year
- Release Month
- Release Month Name
- Release Quarter
- Release Day of Week

These features will be used to analyze seasonal trends and identify optimal release periods.

In [26]:
# Create new features from the Release Date

df["Release Year"] = df["Release Date"].dt.year

df["Release Month"] = df["Release Date"].dt.month

df["Release Month Name"] = df["Release Date"].dt.month_name()

df["Release Quarter"] = df["Release Date"].dt.quarter

df["Release Weekday"] = df["Release Date"].dt.day_name()

# Preview the new columns
df[
    [
        "Release Date",
        "Release Year",
        "Release Month",
        "Release Month Name",
        "Release Quarter",
        "Release Weekday",
    ]
].head()

,Release Date,Release Year,Release Month,Release Month Name,Release Quarter,Release Weekday
0,2023-03-23,2023.0,3.0,March,1.0,Thursday
1,2023-01-05,2023.0,1.0,January,1.0,Thursday
2,2022-12-30,2022.0,12.0,December,4.0,Friday
3,2022-11-23,2022.0,11.0,November,4.0,Wednesday
4,2023-05-04,2023.0,5.0,May,2.0,Thursday


## Executive KPI Dashboard

Before exploring specific business questions, key performance indicators (KPIs) are calculated to provide a high-level overview of Netflix's content catalog and audience engagement.

These metrics summarize the size of the dataset, overall viewing hours, content distribution, language diversity, and global availability.

In [27]:
# Calculate key performance indicators

total_titles = len(df)

total_hours_viewed = df["Hours Viewed"].sum()

average_hours_viewed = df["Hours Viewed"].mean()

movies = (df["Content Type"] == "Movie").sum()

shows = (df["Content Type"] == "Show").sum()

languages = df["Language Indicator"].nunique()

global_titles = (df["Available Globally?"] == "Yes").sum()

global_percentage = (global_titles / total_titles) * 100

print("=" * 60)
print("NETFLIX CONTENT KPI DASHBOARD")
print("=" * 60)

print(f"Total Titles               : {total_titles:,}")

print(f"Total Hours Viewed         : {total_hours_viewed:,.0f}")

print(f"Average Hours Viewed       : {average_hours_viewed:,.0f}")

print(f"Movies                     : {movies:,}")

print(f"TV Shows                   : {shows:,}")

print(f"Languages                  : {languages}")

print(f"Globally Available Titles  : {global_titles:,}")

print(f"Global Availability        : {global_percentage:.1f}%")

NETFLIX CONTENT KPI DASHBOARD
Total Titles               : 24,345
Total Hours Viewed         : 157,970,700,000
Average Hours Viewed       : 6,488,835
Movies                     : 13,908
TV Shows                   : 10,437
Languages                  : 6
Globally Available Titles  : 7,470
Global Availability        : 30.7%


## Business Question 1: Which Content Type Generates the Highest Audience Engagement?

Netflix offers both Movies and TV Shows, but their contribution to total audience engagement may differ significantly.

This analysis compares the total and average viewing hours across content types to understand which format delivers greater audience engagement.

In [28]:
# Compare audience engagement by content type

content_performance = (
    df.groupby("Content Type")["Hours Viewed"]
      .agg(
          Total_Hours_Viewed="sum",
          Average_Hours_Viewed="mean",
          Number_of_Titles="count"
      )
      .reset_index()
)

content_performance

,Content Type,Total_Hours_Viewed,Average_Hours_Viewed,Number_of_Titles
0,Movie,50543800000,3.634153e+06,13908
1,Show,107426900000,1.029289e+07,10437


### Visualization: Audience Engagement by Content Type

The following visualization compares the total audience engagement generated by Movies and TV Shows based on cumulative viewing hours.

In [29]:
# Visualize total viewing hours by content type

fig = px.bar(
    content_performance,
    x="Content Type",
    y="Total_Hours_Viewed",
    color="Content Type",
    text="Total_Hours_Viewed",
    title="Total Hours Viewed by Content Type",
    labels={
        "Content Type": "Content Type",
        "Total_Hours_Viewed": "Total Hours Viewed"
    }
)

fig.update_traces(
    texttemplate="%{text:.2s}",
    textposition="outside"
)

fig.update_layout(
    template="plotly_white",
    showlegend=False,
    title_x=0.5
)

fig.show()

### Business Insight

TV Shows generated approximately **107.4 billion viewing hours**, more than double the **50.5 billion hours** accumulated by Movies.

Although Movies represent a larger share of Netflix's catalog, TV Shows attract substantially higher audience engagement, suggesting that serialized content plays a critical role in maximizing watch time and viewer retention.

### Business Recommendation

Given the significantly higher audience engagement generated by TV Shows, Netflix should continue prioritizing investments in high-quality serialized content to maximize viewer watch time and retention. However, Movies remain an important part of the catalog and should be strategically produced to complement the platform's content portfolio.

## Business Question 2: Which Languages Generate the Highest Audience Engagement?

As a global streaming platform, Netflix offers content in multiple languages. Understanding which languages generate the highest audience engagement can help guide investment decisions, content acquisition, and localization strategies.

This analysis compares the total viewing hours across different language categories to identify the most engaged audiences.

In [30]:
# Analyze audience engagement by language

language_performance = (
    df.groupby("Language Indicator")["Hours Viewed"]
      .agg(
          Total_Hours_Viewed="sum",
          Average_Hours_Viewed="mean",
          Number_of_Titles="count"
      )
      .sort_values(by="Total_Hours_Viewed", ascending=False)
      .reset_index()
)

language_performance

,Language Indicator,Total_Hours_Viewed,Average_Hours_Viewed,Number_of_Titles
0,English,124092000000,7.324519e+06,16942
1,Korean,15371100000,9.815517e+06,1566
2,Non-English,10414700000,3.250531e+06,3204
3,Japanese,7063300000,3.167399e+06,2230
4,Hindi,915000000,2.513736e+06,364
5,Russian,114600000,2.938462e+06,39


In [31]:
# Visualize total viewing hours by language

fig = px.bar(
    language_performance,
    x="Language Indicator",
    y="Total_Hours_Viewed",
    color="Language Indicator",
    text="Total_Hours_Viewed",
    title="Total Hours Viewed by Language",
    labels={
        "Language Indicator": "Language",
        "Total_Hours_Viewed": "Total Hours Viewed"
    }
)

fig.update_traces(
    texttemplate="%{text:.2s}",
    textposition="outside"
)

fig.update_layout(
    template="plotly_white",
    title_x=0.5,
    showlegend=False,
    xaxis_title="Language",
    yaxis_title="Total Hours Viewed"
)

fig.show()

### Business Insight

English-language content generated the highest audience engagement, accounting for **124.1 billion viewing hours**, making it the dominant language on the platform.

Among non-English content, **Korean titles stand out with 15.4 billion viewing hours despite representing only 1,566 titles**, highlighting the strong global appeal of Korean entertainment. This suggests that strategically investing in high-performing international markets can significantly expand audience engagement.

### Business Recommendation

While English content should remain a core investment area, Netflix should continue expanding its portfolio of high-performing international content, particularly Korean productions. Success in regional markets demonstrates that compelling local stories can achieve substantial global viewership and diversify the platform's audience.

## Business Question 3: Does Global Availability Influence Audience Engagement?

Netflix distributes some titles globally, while others are available only in selected regions. Understanding whether globally available content attracts higher audience engagement can help inform future content distribution and licensing strategies.

This analysis compares the total and average viewing hours of globally available and region-specific titles.

In [32]:
# Compare audience engagement based on global availability

global_performance = (
    df.groupby("Available Globally?")["Hours Viewed"]
      .agg(
          Total_Hours_Viewed="sum",
          Average_Hours_Viewed="mean",
          Number_of_Titles="count"
      )
      .reset_index()
)

global_performance

,Available Globally?,Total_Hours_Viewed,Average_Hours_Viewed,Number_of_Titles
0,No,72897000000,4.319822e+06,16875
1,Yes,85073700000,1.138871e+07,7470


In [33]:
# Visualize total viewing hours by global availability

fig = px.bar(
    global_performance,
    x="Available Globally?",
    y="Total_Hours_Viewed",
    color="Available Globally?",
    text="Total_Hours_Viewed",
    title="Total Hours Viewed by Global Availability",
    labels={
        "Available Globally?": "Available Globally",
        "Total_Hours_Viewed": "Total Hours Viewed"
    }
)

fig.update_traces(
    texttemplate="%{text:.2s}",
    textposition="outside"
)

fig.update_layout(
    template="plotly_white",
    title_x=0.5,
    showlegend=False,
    xaxis_title="Available Globally",
    yaxis_title="Total Hours Viewed"
)

fig.show()

### Business Insight

Globally available titles generated approximately **85.1 billion viewing hours**, exceeding the **72.9 billion hours** accumulated by region-specific titles despite representing less than one-third of the catalog.

On average, globally available titles received **11.4 million viewing hours per title**, compared with **4.3 million** for non-global titles. This suggests that broader distribution substantially increases audience reach and engagement.

### Business Recommendation

Netflix should prioritize global distribution for high-potential original content whenever licensing agreements permit. Expanding worldwide availability can significantly increase audience reach, improve content visibility, and maximize overall viewer engagement.

## Business Question 4: How Has Netflix's Content Release Strategy Evolved Over Time?

Understanding release trends over the years helps identify how Netflix has expanded its content library and whether its content production strategy has changed over time.

This analysis examines the number of titles released each year to identify long-term growth patterns and shifts in content investment.

In [34]:
# Number of titles released each year

yearly_releases = (
    df.groupby("Release Year")
      .size()
      .reset_index(name="Number_of_Titles")
)

yearly_releases

,Release Year,Number_of_Titles
0,2010.0,16
1,2011.0,6
2,2012.0,2
3,2013.0,21
4,2014.0,47
...,...,...
9,2019.0,1109
10,2020.0,1218
11,2021.0,1255
12,2022.0,1532


In [35]:
# Visualize Netflix's content releases over time

fig = px.line(
    yearly_releases,
    x="Release Year",
    y="Number_of_Titles",
    markers=True,
    title="Netflix Content Releases by Year",
    labels={
        "Release Year": "Release Year",
        "Number_of_Titles": "Number of Titles"
    }
)

fig.update_layout(
    template="plotly_white",
    title_x=0.5,
    xaxis=dict(dtick=1)
)

fig.show()

### Business Insight

Netflix's content production increased steadily from **2013 onward**, with particularly rapid growth between **2017 and 2022**. The platform reached its highest level of content releases in **2022**, reflecting a significant expansion in its content investment strategy.

A noticeable decline is observed in **2023**. Since the dataset may not represent the complete year, this decrease should be interpreted cautiously rather than as evidence of reduced content production.

### Business Recommendation

Netflix should continue balancing consistent content releases with investments in high-quality productions. Monitoring annual release trends alongside audience engagement can help determine whether increasing the number of releases continues to generate proportional growth in viewer engagement.

## Business Question 5: Which Release Years Generated the Highest Audience Engagement?

Releasing more titles does not necessarily lead to higher audience engagement. This analysis evaluates whether content released in certain years attracted significantly more viewing hours than others.

The findings can help identify periods in which Netflix's content strategy was most effective at capturing audience attention.

In [36]:
# Analyze audience engagement by release year

yearly_engagement = (
    df.groupby("Release Year")["Hours Viewed"]
      .agg(
          Total_Hours_Viewed="sum",
          Average_Hours_Viewed="mean",
          Number_of_Titles="count"
      )
      .reset_index()
      .sort_values(by="Release Year")
)

yearly_engagement

,Release Year,Total_Hours_Viewed,Average_Hours_Viewed,Number_of_Titles
0,2010.0,125500000,7.843750e+06,16
1,2011.0,155500000,2.591667e+07,6
2,2012.0,6300000,3.150000e+06,2
3,2013.0,260600000,1.240952e+07,21
4,2014.0,507700000,1.080213e+07,47
...,...,...,...,...
9,2019.0,6373100000,5.746709e+06,1109
10,2020.0,8181700000,6.717323e+06,1218
11,2021.0,10293200000,8.201753e+06,1255
12,2022.0,18475900000,1.205999e+07,1532


In [37]:
# Visualize audience engagement by release year

fig = px.line(
    yearly_engagement,
    x="Release Year",
    y="Total_Hours_Viewed",
    markers=True,
    title="Total Hours Viewed by Release Year",
    labels={
        "Release Year": "Release Year",
        "Total_Hours_Viewed": "Total Hours Viewed"
    }
)

fig.update_layout(
    template="plotly_white",
    title_x=0.5,
    xaxis=dict(dtick=1)
)

fig.show()

### Business Recommendation

Netflix should continue investing in high-quality new releases while regularly evaluating whether increased content production translates into sustained audience engagement. Future release strategies should prioritize content performance and viewer demand rather than focusing solely on increasing the number of titles.

## Business Question 6: Which Months Are Best for Releasing Content?

The timing of a content release can influence its visibility and audience engagement. Identifying months that consistently generate higher viewing hours can help Netflix optimize its content release calendar.

This analysis compares total audience engagement across different release months to identify seasonal release patterns.

In [38]:
# Analyze audience engagement by release month

monthly_engagement = (
    df.groupby(["Release Month", "Release Month Name"])["Hours Viewed"]
      .agg(
          Total_Hours_Viewed="sum",
          Average_Hours_Viewed="mean",
          Number_of_Titles="count"
      )
      .reset_index()
      .sort_values("Release Month")
)

monthly_engagement

,Release Month,Release Month Name,Total_Hours_Viewed,Average_Hours_Viewed,Number_of_Titles
0,1.0,January,7265400000,1.214950e+07,598
1,2.0,February,7096000000,1.306814e+07,543
2,3.0,March,7425200000,1.095162e+07,678
3,4.0,April,6858600000,1.088667e+07,630
4,5.0,May,7068000000,1.156792e+07,611
...,...,...,...,...,...
7,8.0,August,6787800000,1.034726e+07,656
8,9.0,September,7254400000,9.992287e+06,726
9,10.0,October,8104700000,1.031132e+07,786
10,11.0,November,7739100000,1.076370e+07,719


In [39]:
# Visualize audience engagement by release month

fig = px.bar(
    monthly_engagement,
    x="Release Month Name",
    y="Total_Hours_Viewed",
    color="Release Month Name",
    text="Total_Hours_Viewed",
    title="Total Hours Viewed by Release Month",
    labels={
        "Release Month Name": "Release Month",
        "Total_Hours_Viewed": "Total Hours Viewed"
    }
)

fig.update_traces(
    texttemplate="%{text:.2s}",
    textposition="outside"
)

fig.update_layout(
    template="plotly_white",
    title_x=0.5,
    showlegend=False,
    xaxis_title="Release Month",
    yaxis_title="Total Hours Viewed"
)

fig.show()


### Business Insight

Content released in **December** generated the highest audience engagement, with approximately **10.0 billion viewing hours**, followed by **June** with **8.5 billion viewing hours**. This suggests that audiences are particularly active during holiday periods and mid-year release windows.

Although engagement varies across months, the differences are relatively moderate, indicating that successful content performance depends on both release timing and content quality.

### Business Recommendation

Netflix should prioritize releasing its most anticipated titles during high-engagement periods such as **June** and **December**, when audience activity appears strongest. However, release timing should complement—rather than replace—investments in high-quality content and effective marketing campaigns.

## Business Question 7: Which Weekdays Generate the Highest Audience Engagement?

Release timing extends beyond months to the specific day of the week. Identifying whether certain weekdays consistently attract higher audience engagement can help Netflix optimize its content release schedule.

This analysis compares total viewing hours across weekdays to uncover potential release-day patterns.

In [40]:
# Analyze audience engagement by release weekday

weekday_order = [
    "Monday", "Tuesday", "Wednesday",
    "Thursday", "Friday", "Saturday", "Sunday"
]

weekday_engagement = (
    df.groupby("Release Weekday")["Hours Viewed"]
      .agg(
          Total_Hours_Viewed="sum",
          Average_Hours_Viewed="mean",
          Number_of_Titles="count"
      )
      .reindex(weekday_order)
      .reset_index()
)

weekday_engagement

,Release Weekday,Total_Hours_Viewed,Average_Hours_Viewed,Number_of_Titles
0,Monday,3950800000,9.274178e+06,426
1,Tuesday,5535600000,5.863983e+06,944
2,Wednesday,15724400000,1.224642e+07,1284
3,Thursday,20268100000,1.798412e+07,1127
4,Friday,38135200000,1.006471e+07,3789
5,Saturday,5119500000,2.216234e+07,231
6,Sunday,1934200000,1.111609e+07,174


In [41]:
# Visualize audience engagement by release weekday

fig = px.bar(
    weekday_engagement,
    x="Release Weekday",
    y="Total_Hours_Viewed",
    color="Release Weekday",
    text="Total_Hours_Viewed",
    title="Total Hours Viewed by Release Weekday",
    labels={
        "Release Weekday": "Release Weekday",
        "Total_Hours_Viewed": "Total Hours Viewed"
    }
)

fig.update_traces(
    texttemplate="%{text:.2s}",
    textposition="outside"
)

fig.update_layout(
    template="plotly_white",
    title_x=0.5,
    showlegend=False,
    xaxis_title="Release Weekday",
    yaxis_title="Total Hours Viewed"
)

fig.show()

### Business Insight

Content released on **Fridays** generated the highest audience engagement, accumulating approximately **38.1 billion viewing hours**, significantly outperforming releases on other weekdays. This aligns with typical consumer viewing behavior, as audiences have more leisure time heading into the weekend.

Although **Saturday** shows the highest average viewing hours per title, the relatively small number of releases on that day suggests this result should be interpreted cautiously.

### Business Recommendation

Netflix should continue prioritizing major content releases on **Fridays** to maximize audience reach and capitalize on increased weekend viewing activity. However, release-day decisions should also consider content type, target audience, and marketing strategy to optimize overall performance.

## Business Question 8: Which Release Quarters Generate the Highest Audience Engagement?

Analyzing audience engagement by quarter helps identify broader seasonal trends beyond individual months. These insights can support long-term content planning and scheduling decisions.

This analysis compares total viewing hours across the four calendar quarters.

In [42]:
# Analyze audience engagement by release quarter

quarterly_engagement = (
    df.groupby("Release Quarter")["Hours Viewed"]
      .agg(
          Total_Hours_Viewed="sum",
          Average_Hours_Viewed="mean",
          Number_of_Titles="count"
      )
      .reset_index()
      .sort_values("Release Quarter")
)

quarterly_engagement

,Release Quarter,Total_Hours_Viewed,Average_Hours_Viewed,Number_of_Titles
0,1.0,21786600000,1.197724e+07,1819
1,2.0,22437700000,1.183423e+07,1896
2,3.0,20555200000,1.028789e+07,1998
3,4.0,25888300000,1.144487e+07,2262


In [43]:
# Convert quarter numbers to labels (Q1, Q2, Q3, Q4)
quarterly_engagement["Release Quarter"] = (
    quarterly_engagement["Release Quarter"]
    .astype(int)
    .astype(str)
    .radd("Q")
)

# Visualize audience engagement by release quarter
fig = px.bar(
    quarterly_engagement,
    x="Release Quarter",
    y="Total_Hours_Viewed",
    color="Release Quarter",
    text="Total_Hours_Viewed",
    title="Total Hours Viewed by Release Quarter",
    labels={
        "Release Quarter": "Release Quarter",
        "Total_Hours_Viewed": "Total Hours Viewed"
    }
)

fig.update_traces(
    texttemplate="%{text:.2s}",
    textposition="outside"
)

fig.update_layout(
    template="plotly_white",
    title_x=0.5,
    showlegend=False,
    xaxis_title="Release Quarter",
    yaxis_title="Total Hours Viewed"
)

fig.show()

### Business Insight

The fourth quarter (**Q4**) generated the highest audience engagement, accumulating approximately **25.9 billion viewing hours**, outperforming all other quarters. This suggests that content released during the final months of the year benefits from increased viewer activity, likely influenced by holiday periods and extended leisure time.

Although engagement remains relatively consistent across the first three quarters, the noticeable increase in Q4 highlights a valuable seasonal opportunity for major content releases.

### Business Recommendation

Netflix should prioritize launching its highest-profile content during **Q4** to capitalize on peak audience engagement. However, maintaining a balanced release schedule throughout the year remains important to sustain subscriber interest and avoid concentrating all major releases within a single quarter.

## Key Drivers of Audience Engagement

Based on the analyses conducted throughout this project, several clear patterns emerged regarding the factors associated with higher audience engagement on Netflix.

### Key Findings

- **TV Shows** generated substantially higher viewing hours than Movies, indicating stronger long-term audience engagement.
- **English-language** content accounted for the largest share of viewing hours, while **Korean content** demonstrated exceptional performance despite a smaller catalog.
- **Globally available titles** consistently attracted higher total and average viewing hours than region-specific content.
- Audience engagement increased significantly for **recent releases**, highlighting the growing popularity of newer content.
- **Fridays**, **December**, and **Q4** emerged as the strongest release periods, suggesting that release timing plays an important role in maximizing audience reach.

These findings indicate that successful audience engagement is influenced by a combination of **content type, language, global accessibility, release timing, and strategic distribution** rather than a single factor.

# Executive Summary

This project analyzed Netflix content performance using viewing-hour data to identify the factors that drive audience engagement.

The analysis revealed that **TV Shows consistently outperform Movies**, **English-language content dominates overall engagement**, and **Korean content delivers outstanding performance relative to its catalog size**. Titles released globally also attract significantly higher viewing hours than region-specific releases, emphasizing the value of worldwide distribution.

Seasonality plays an important role in content success. **Friday releases**, **December launches**, and **Q4 releases** generated the strongest audience engagement, suggesting that strategic release timing can improve content visibility and viewer reach.

### Strategic Recommendations

- Increase investment in high-performing serialized content.
- Continue expanding globally accessible original programming.
- Strengthen investment in successful international markets, particularly Korean content.
- Schedule flagship releases during high-engagement periods such as Fridays and Q4.
- Combine release timing with high-quality content and effective marketing to maximize audience engagement.

Overall, the analysis demonstrates how data-driven insights can support strategic decision-making in content planning, distribution, and audience growth.